This experiment runs strong scaling evaluation on a dragonfly network. 

# 0. User Parameters, Imports, Environment

## User Parameters

In [1]:

# Path to the directory containing the merlin benchmark workflow and examples
workflow_dir = '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/'


# Path to the directory where the output of the experiment runs will be stored
experiment_name = 'distribution_verification'

import os
import sys
sys.path.append(workflow_dir)
import workflow_args
from workflow_utils import *

# Modify the following run spec generation to change the test case being verified
run_specs = []

node_counts = [1, 2, 4, 7, 8, 16]
ranks_per_node_counts = [1, 3, 5, 8]

# We make run specs for both dragonfly and fattree topologies

# For dragonfly, make sure there are enough groups that we have at least 1 group per rank
num_groups = max(node_counts) * max(ranks_per_node_counts)
endpoint_params = {
    'tg_output_file_name': 'check.log'
}
dragonfly_specs = workflow_args.generate_merlin_run_specs(
    node_counts=node_counts,
    ranks_per_node=ranks_per_node_counts,
    topologies = ['dragonfly'],
    global_params={
        'stop_at': (200),
        'verbose': 5
    },
    endpoint_params=endpoint_params,
    dragonfly_params = {
        'dragonfly_num_groups': [num_groups],
        'dragonfly_routers_per_group': [8],
        'dragonfly_hosts_per_router': [8],

    },
    experiment_name=experiment_name
)

fattree_specs = workflow_args.generate_merlin_run_specs(
    node_counts=node_counts,
    ranks_per_node=ranks_per_node_counts,
    topologies = ['fattree'],
    global_params={
        'stop_at': (200),
        'verbose': 5
    },
    endpoint_params=endpoint_params,
    fattree_params = {
        'fattree_shape': '4,4:4,4:4,4:7'
    },
    experiment_name=experiment_name
)

run_specs = dragonfly_specs + fattree_specs



# Whether to force re-running experiments even if they have already been completed
# If set to 0, existing results will be used instead of rerunning experiments
FORCED = 0

# Path to the directory containing the apptainer executable
apptainer_bin_path = '/lus/scratch/crickett/software/usr/bin'

# Path where apptainer should store its cache
apptainer_cache_dir = '/lus/bnchlu1/neth/.local/apptainer/'

# URL of the SST container to be used for the workflow and the destination path where it should be saved.
# If the destination path is already created, the container will not be downloaded and the url will be ignored.
sst_container_url = 'ghcr.io/hpc-ai-adv-dev/sst-perf-track-full:16.0.0'
sst_container_dest_name = 'sst-perf-track-full.sif'


print(f'workflow_dir: {workflow_dir}')

workflow_dir: /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/


## Calculated 'Constants'

In [2]:
# Full path to the SST input configuration file
sst_input_config = os.path.join(workflow_dir, 'merlin_benchmark.py')

# Directory containing per-simulation subdirectories
output_dir = os.path.join(workflow_dir, experiment_name + '_out')

# Full path to where the final .sif containerfile will live
sst_container_dest_path = os.path.join(workflow_dir, sst_container_dest_name)

# CSV file containing the consolidated experiment results after all simulations
# have been completed
experiment_result_file = os.path.join(workflow_dir, f'{experiment_name}_results.csv')

sst_input_config
sst_container_dest_path

'/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif'

## Imports

In [3]:


import workflow_args
import json
import glob
import humanfriendly
import pandas as pd
from plotnine import *

## Environment + Checks

In [4]:
os.environ["PATH"] += os.pathsep + apptainer_bin_path
#os.environ["PATH"] += os.pathsep + e4s_cl_path
os.environ["LANG"] = "C.UTF-8"
os.environ["LC_ALL"] = "C.UTF-8"
os.environ['APPTAINER_CACHEDIR'] = apptainer_cache_dir

# 1. Download and Prepare Containers

## Download containers, create `.sif`

In [5]:
cd(workflow_dir)

get_convert_to_sif()

# script expects there not to be an extension on the output file name
if os.path.exists(sst_container_dest_path):
    print(f"{sst_container_dest_path} already exists.")
else:
    run_cmd(f"./convert-to-sif.sh {sst_container_url} -o {workflow_dir} -f {sst_container_dest_name.replace('.sif', '')}")



> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/
/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif already exists.


## Prepare the e4s-cl profile

In [6]:
run_cmd(f"e4s-cl profile edit --add-files {workflow_dir}")
run_cmd(f'e4s-cl profile edit --add-files {workflow_dir}/*.py')
run_cmd(f'e4s-cl profile edit --add-files {workflow_dir}/*.so')
run_cmd(f'e4s-cl profile edit --image {sst_container_dest_path}')

> e4s-cl profile edit --add-files /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/


File /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark already in profile's files


> e4s-cl profile edit --add-files /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark//*.py


File /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/*.py already in profile's files


> e4s-cl profile edit --add-files /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark//*.so


File /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/*.so already in profile's files


> e4s-cl profile edit --image /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif


## Check that the profile injects a host MPI

If the selected e4s-cl profile has no MPI library bound, e4s-cl performs no MPI
substitution. The container's own MPI then finds no process manager and falls
back to *singleton init*, so every task silently becomes rank 0 of a 1-rank job
instead of one N-rank simulation.


In [7]:
check_mpi_binding(sst_container_dest_path)

e4s-cl profile : sst-experiments
profile image  : /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif
container SST  : /opt/SST/16.0.0/libexec/sstsim.x
container MPI  : libmpi.so.12
bound host MPI : libmpi_cray.so.12 -> /opt/cray/pe/lib64/libmpi_cray.so.12
bound host PMI : libpals.so.0, libpmi.so.0, libpmi2.so.0
OK: e4s-cl will inject a host MPI into the container.


True

# 2. Build Benchmarks

In [8]:
# Run Make within the container
# We have to create and mount this .conf file for SST to use when running
run_cmd('touch sstsimulator.conf')
run_in_container('make clean', sst_container_dest_path, additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
run_in_container('make -j10', sst_container_dest_path, additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')

> touch sstsimulator.conf
> ['apptainer', 'exec', '--no-home', '--bind', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark:/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--pwd', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--bind', 'sstsimulator.conf:/home/users/neth/.sst/sstsimulator.conf', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/sst-perf-track-full.sif', 'bash', '-lc', 'make clean']
rm -rf libmerlin_benchmark.so libmerlin_benchmark_test.so
rm -f /lus/bnchlu1/neth/mysst/sst-elements/install/lib/sst-elements-library/libmerlin_benchmark.so
	Model merlin_benchmark has been unregistered!
	Model merlin_benchmark_test not found

rm -f *.json *.tmp *.time *.out *.csv
> ['apptainer', 'exec', '--no-home', '--bind', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark:/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--pwd', '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark', '--bind', 'sstsimulator.conf:/home/users/neth/.sst/sstsimulator.conf', '/lus/bnchlu1/neth/s

	Error removing /home/users/neth/.sst/sstsimulator.conf before moving updated config
	Error removing /home/users/neth/.sst/sstsimulator.conf before moving updated config


g++ -std=c++17  -std=c++17 -fexcess-precision=fast -fPIC -DHAVE_CONFIG_H -I/opt/SST/16.0.0/include -g -shared -fno-common -Wl,-undefined -Wl,dynamic_lookup "-DENABLE_SSTCHECKPOINT" -o libmerlin_benchmark.so src/trafficgen.cc
/opt/SST/16.0.0/bin//sst-register merlin_benchmark merlin_benchmark_LIBDIR=/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark


# 3. Launch Runs

In [ ]:
launch_jobs(run_specs, workflow_dir, output_dir, sst_input_config, experiment_name, FORCED)

> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/distribution_verification_out/distribution_verification_dragonfly_n1_r1_4f451b8813_t1
Skipping distribution_verification_dragonfly_n1_r1_4f451b8813_t1 as it is already COMPLETED
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/distribution_verification_out/distribution_verification_dragonfly_n1_r3_58741c6462_t1
Skipping distribution_verification_dragonfly_n1_r3_58741c6462_t1 as it is already COMPLETED
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/distribution_verification_out/distribution_verification_dragonfly_n1_r5_e8ff4e7b8b_t1
Skipping distribution_verification_dragonfly_n1_r5_e8ff4e7b8b_t1 as it is already COMPLETED
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/distribution_verification_out/distribution_verification_dragonfly_n1_r8_9f989c44ae_t1
Skipping distribution_verification_dragonfly_n1_r8_9f989c44ae_t1 as it is already COMPLETED
> cd /lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/distribution_

# 4. Wait for Job Completion

In [12]:
wait_for_jobs(job_name=experiment_name)

# 5. Verify Results

In [5]:
from collections import Counter
import glob as glob_module
import re

param_file_paths = sorted(glob_module.glob(os.path.join(output_dir, '**', 'params.json'), recursive=True))
check_log_rows = []

for param_file_path in param_file_paths:
    run_dir = os.path.dirname(param_file_path)
    if get_run_status(run_dir) != 'COMPLETED':
        print(f"Run not completed for {run_dir}")
        continue

    row = {}
    with open(param_file_path, 'r') as params_file:
        row.update(json.load(params_file))

    check_file_path = param_file_path.replace('params.json', 'check.log')
    if row['node_count'] != 1 or row['ranks_per_node'] == 1:
        merge_count = merge_rank_log_files(check_file_path)
        if merge_count == 0:
            continue
    row['check.log'] = check_file_path
    check_log_rows.append(row)

check_log_df = pd.DataFrame(check_log_rows)
check_log_df

,run_id,run_name,experiment_name,topology,trial,node_count,ranks_per_node,srun_command,mpirun_command,stop_at,...,tg_packet_delay_exponential_lambda,dragonfly_hosts_per_router,dragonfly_routers_per_group,dragonfly_intergroup_links,dragonfly_num_groups,dragonfly_algorithm,dragonfly_adaptive_threshold,dragonfly_global_routing_mode,check.log,fattree_shape
0,58741c6462,distribution_verification_dragonfly_n1_r3_5874...,distribution_verification,dragonfly,1,1,3,srun --nodes=1 --ntasks-per-node=3 sst --timin...,mpirun -n 3 -N 3 sst --timing-info=3 --paralle...,200ns,...,None,8.0,8.0,1.0,128.0,minimal,2.0,absolute,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,NaN
1,e8ff4e7b8b,distribution_verification_dragonfly_n1_r5_e8ff...,distribution_verification,dragonfly,1,1,5,srun --nodes=1 --ntasks-per-node=5 sst --timin...,mpirun -n 5 -N 5 sst --timing-info=3 --paralle...,200ns,...,None,8.0,8.0,1.0,128.0,minimal,2.0,absolute,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,NaN
2,9f989c44ae,distribution_verification_dragonfly_n1_r8_9f98...,distribution_verification,dragonfly,1,1,8,srun --nodes=1 --ntasks-per-node=8 sst --timin...,mpirun -n 8 -N 8 sst --timing-info=3 --paralle...,200ns,...,None,8.0,8.0,1.0,128.0,minimal,2.0,absolute,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,NaN
3,6b1293c19a,distribution_verification_fattree_n1_r3_6b1293...,distribution_verification,fattree,1,1,3,srun --nodes=1 --ntasks-per-node=3 sst --timin...,mpirun -n 3 -N 3 sst --timing-info=3 --paralle...,200ns,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,"4,4:4,4:4,4:7"
4,abb6c1a0ec,distribution_verification_fattree_n1_r5_abb6c1...,distribution_verification,fattree,1,1,5,srun --nodes=1 --ntasks-per-node=5 sst --timin...,mpirun -n 5 -N 5 sst --timing-info=3 --paralle...,200ns,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,"4,4:4,4:4,4:7"
5,652e92d8d2,distribution_verification_fattree_n1_r8_652e92...,distribution_verification,fattree,1,1,8,srun --nodes=1 --ntasks-per-node=8 sst --timin...,mpirun -n 8 -N 8 sst --timing-info=3 --paralle...,200ns,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/lus/bnchlu1/neth/sst-benchmarks/merlin_benchm...,"4,4:4,4:4,4:7"


In [7]:
import difflib


def sort_lines(check_log_path):
    with open(check_log_path, 'r') as run_log_file:
        return sorted(run_log_file.readlines())

resource_columns = {
    'check.log', 'run_id', 'run_name', 'trial', 'node_count', 'ranks_per_node',
    'srun_command', 'mpirun_command',
}
group_columns = ['topology', 'dragonfly_num_groups', 'dragonfly_routers_per_group', 'dragonfly_hosts_per_router', 'fattree_shape']

#intersect resource and group columns with dataframe
resource_columns = resource_columns.intersection(check_log_df.columns)
group_columns = [col for col in group_columns if col in check_log_df.columns]


failed_runs = []
groups = check_log_df.groupby(group_columns, dropna=False, sort=False)
for group_key, group in groups:
    print("Checking group: ", dict(zip(group_columns, group_key)))
    runs_by_resource = group.sort_values(['node_count', 'ranks_per_node', 'check.log'], kind='stable')
    reference_file = runs_by_resource.iloc[0]['check.log']

    reference_lines = sort_lines(reference_file)
    for _, candidate in runs_by_resource.iloc[1:].iterrows():
        candidate_file = candidate['check.log']
        candidate_lines = sort_lines(candidate_file)
        diff_filename = f"{candidate_file}.diff"
        if reference_lines != candidate_lines:
            print(f"Difference found for candidate: {candidate_file}")
            failed_runs.append(candidate_file)
            with open(diff_filename, 'w') as diff_file:
                diff_file.writelines([f"{line}\n" for line in difflib.unified_diff(reference_lines, candidate_lines)]) 
        
if len(failed_runs) > 0:
    print("Some runs failed.")
else:
    print("All runs passed.")
for failed_run in failed_runs:
    print(failed_run)


Checking group:  {'topology': 'dragonfly', 'dragonfly_num_groups': 128.0, 'dragonfly_routers_per_group': 8.0, 'dragonfly_hosts_per_router': 8.0, 'fattree_shape': nan}
Checking group:  {'topology': 'fattree', 'dragonfly_num_groups': nan, 'dragonfly_routers_per_group': nan, 'dragonfly_hosts_per_router': nan, 'fattree_shape': '4,4:4,4:4,4:7'}
All runs passed.
